In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/albin/egna_proj/block_puzzle_rl/

/home/albin/egna_proj/block_puzzle_rl


/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
# ⬇️ SB3 + Gym imports
import numpy as np
import torch
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from gymnasium import spaces
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np
from collections import deque
from stable_baselines3.common.callbacks import CallbackList
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
import os
from datetime import datetime
# ⬇️ Your imports
from game.block_puzzle_env import BlockPuzzleEnv
from agent.utils import encode_state


In [3]:

# ⬇️ Custom wrapper to convert dict obs → flat array
class FlattenedBlockEnv(gym.Env):
    def __init__(self, width=12, height=12, num_blocks=3):
        super().__init__()
        self.raw_env = BlockPuzzleEnv(width, height, num_blocks)
        self.action_space = self.raw_env.action_space
        dummy_obs, _ = self.raw_env.reset()
        sample_obs = encode_state(dummy_obs)
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=sample_obs.shape, dtype=np.float32
        )
    
    def reset(self, seed=None, options=None):
        obs_dict, _ = self.raw_env.reset()
        flat_obs = encode_state(obs_dict)
        return flat_obs.astype(np.float32), {}

    def step(self, action):
        # Handle batched action from DummyVecEnv (e.g., [0, 9, 10])
        if isinstance(action, (list, np.ndarray)) and len(action) == 3:
            a0, a1, a2 = map(int, action)
        else:
            # Flat index → unravel into (block_index, row, col)
            a0, a1, a2 = np.unravel_index(action, self.action_space.nvec)

        raw_action = (a0, a1, a2)
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(raw_action)
        flat_obs = encode_state(obs_dict)
        return flat_obs.astype(np.float32), reward, terminated, truncated, info


    def render(self):
        return self.raw_env.render()

    def close(self):
        return self.raw_env.close()


class DiscreteActionWrapper(gym.Env):
    def __init__(self, raw_env):
        super().__init__()
        self.raw_env = raw_env
        self.original_action_space = raw_env.action_space  # Should be MultiDiscrete
        self.obs_space = self.raw_env.observation_space

        # Flatten MultiDiscrete([a, b, c]) → Discrete(a * b * c)
        self.action_space = spaces.Discrete(np.prod(self.original_action_space.nvec))
        self.observation_space = spaces.Box(low=0, high=1, shape=(encode_state(self.raw_env.reset()[0]).shape[0],), dtype=np.float32)

    def reset(self, **kwargs):
        obs_dict, _ = self.raw_env.reset(**kwargs)
        return encode_state(obs_dict).astype(np.float32), {}

    def step(self, flat_action):
        a0, a1, a2 = np.unravel_index(flat_action, self.original_action_space.nvec)
        action = (int(a0), int(a1), int(a2))
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(action)
        flat_obs = encode_state(obs_dict)
        return flat_obs.astype(np.float32), reward, terminated, truncated, info

    def render(self, **kwargs):
        return self.raw_env.render(**kwargs)



In [34]:
class AveragedMetricsCallback(BaseCallback):
    def __init__(self, metrics_to_track=None, verbose=0):
        super().__init__(verbose)
        self.metrics_to_track = metrics_to_track or ['cleared_lines', 'invalid_moves', 'move_count']
        self.metric_buffers = {metric: deque(maxlen=100) for metric in self.metrics_to_track}

    def _on_step(self):
        infos = self.locals["infos"]
        for info in infos:
            if "episode" in info:  # Only log at the end of episodes
                for metric in self.metrics_to_track:
                    if metric in info:
                        self.metric_buffers[metric].append(info[metric])

        for metric, buffer in self.metric_buffers.items():
            if buffer:  # Avoid empty
                avg_value = np.mean(buffer)
                self.logger.record(f"custom/{metric}_avg", avg_value)

        return True
    
class SaveEveryNTimestepsCallback(BaseCallback):
    def __init__(self, save_freq: int, save_path: str, verbose=0, name="save_model", with_time=True, with_timesteps=True):
        super().__init__(verbose)
        self.save_freq = save_freq
        self.save_path = save_path
        self.name = name
        self.with_time = with_time
        self.with_timesteps = with_timesteps
        os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self) -> bool:
        if self.num_timesteps % self.save_freq == 0:
            day_time_string = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") if self.with_time else ""
            timesteps_string = f"_{self.num_timesteps}" if self.with_timesteps else ""
            save_file = os.path.join(self.save_path, f"{self.name}_{day_time_string}_{timesteps_string}.zip")
            self.model.save(save_file)
            if self.verbose:
                print(f"Saved model to {save_file}")
        return True

In [6]:
raw_env = BlockPuzzleEnv(width=8, height=10, num_blocks=3)
wrapped_env = DiscreteActionWrapper(raw_env)
def mask_fn(env):
    return env.raw_env.game.compute_action_mask()
masked_env = ActionMasker(wrapped_env, mask_fn)
check_env(masked_env, warn=True)  # ✅ Now this should pass

In [ ]:
def make_env():
    raw_env = BlockPuzzleEnv(width=8, height=10, num_blocks=3)
    wrapped_env = DiscreteActionWrapper(raw_env)

    def mask_fn(env):
        return env.raw_env.game.compute_action_mask()

    masked_env = ActionMasker(wrapped_env, mask_fn)
    monitored_env = Monitor(masked_env)
    return monitored_env

n_envs = 20  # or whatever number you want
vec_env = DummyVecEnv([make_env for _ in range(n_envs)])


model = MaskablePPO(
    policy="MlpPolicy",
    env=vec_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=512,
    gamma=0.99,
    device='cpu',
    verbose=1,
    tensorboard_log="./sb3_logs/"
)

callback_list = CallbackList([
    AveragedMetricsCallback(), 
    SaveEveryNTimestepsCallback(save_freq=250_000, save_path="./sb3_model_saves/", name="sb3_block_ppo_mlp_masked"),
    SaveEveryNTimestepsCallback(save_freq=100_000, save_path="./crash_backup_save/", name="backup_save", with_time=False, with_timesteps=False),
])

load_weights = True
weight_path = "sb3_model_saves/sb3_block_ppo_mlp_masked_750000.zip"
if load_weights:
    model.set_parameters(weight_path, exact_match=True)
        
model.learn(total_timesteps=20_000_000, callback=callback_list, tb_log_name="PPO_MLP_MASKED")
model.save("sb3_block_ppo_mlp_masked")


In [30]:
single_env = vec_env.envs[0]  # Get one of the 20 envs
single_env.env.env.raw_env.game.compute_action_mask().shape

(432,)

In [ ]:
callback_list = CallbackList([
    AveragedMetricsCallback(), 
    SaveEveryNTimestepsCallback(save_freq=1_000_000, save_path="./sb3_model_saves/", name="sb3_block_ppo_mlp_masked_rewards2"),
    SaveEveryNTimestepsCallback(save_freq=100_000, save_path="./crash_backup_save/", name="backup_save", with_time=False, with_timesteps=False),
])

for i in range(0, 50):
    if i != 0:
        backup_path = "crash_backup_save/backup_save__"
        model.set_parameters(backup_path, exact_match=True)

    try:
        model.learn(total_timesteps=30_000_001, callback=callback_list, tb_log_name="PPO_MLP_MASKED")
    except Exception as e:
        pass

In [49]:
from game.plot_game import render_text

In [64]:
obs, _ = single_env.reset()
done = False
total_reward = 0
step = 0

while not done:
    # Get valid action mask
    mask = single_env.env.env.raw_env.game.compute_action_mask()
    # Get action probabilities from the model
    
    render_text(single_env.env.env.raw_env.game.grid.get_game_grid())
    print()
    
    action_probs = model.policy.forward(torch.tensor(obs).unsqueeze(0).to(model.device))[0].detach().cpu().numpy().flatten()
    # Mask invalid actions
    masked_probs = action_probs * mask
    if masked_probs.sum() == 0:
        # If all actions are masked, pick a random valid action
        action = np.random.choice(np.where(mask)[0])
    else:
        # Pick the action with the highest probability among valid actions
        action = np.argmax(masked_probs)
    obs, reward, terminated, truncated, info = single_env.step(action)
    total_reward += reward
    done = terminated or truncated
    step += 1

print(f"Evaluation finished in {step} steps, total reward: {total_reward}")

□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

■ ■ □ □ □ □ □ □ □ □ □ □
□ ■ ■ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

■ ■ ■ ■ ■ □ □ □ □ □ □ □
□ ■ ■ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

■ ■ ■ ■ ■ ■ ■ ■ ■ □ □ □
□ ■ ■ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □